In [1]:
from fredapi import Fred
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()
api_key = os.getenv("FRED_API_KEY")
fred = Fred(api_key=api_key)

series = {
    "unemployment_rate": "UNRATE",
    "labor_force_participation_rate": "CIVPART",
    "initial_jobless_claims": "IC4WSA",
    "nonfarm_payroll_employment": "PAYEMS",
    "employment_population_ratio": "EMRATIO",
    "consumer_price_index": "CPIAUCSL",
    "core_consumer_price_index": "CPILFESL",
    "personal_consumption_expenditures_price_index": "PCEPI",
    "federal_funds_rate": "FEDFUNDS",
    "ten_year_treasury_yield": "DGS10",
    "two_year_treasury_yield": "DGS2",
    "industrial_production_index": "INDPRO",
    "retail_sales": "RSAFS",
    "housing_starts": "HOUST",
    "sp500_index": "SP500",
    "gdp": "GDPC1"
}

Start by configuring FRED API credentials and listing out the features we'll use to model unemployment rate.

In [ ]:
raw_df = pd.DataFrame()

for name, code in series.items():
    s = fred.get_series(code)
    raw_df[name] = s

raw_df.head()

Pull data from FRED API.

In [4]:
raw_df.isna().mean().sort_values(ascending=False)

sp500_index                                      0.918172
initial_jobless_claims                           0.891605
gdp                                              0.667375
two_year_treasury_yield                          0.591923
retail_sales                                     0.561105
ten_year_treasury_yield                          0.476089
housing_starts                                   0.140276
personal_consumption_expenditures_price_index    0.140276
core_consumer_price_index                        0.115834
federal_funds_rate                               0.082891
unemployment_rate                                0.001063
consumer_price_index                             0.001063
labor_force_participation_rate                   0.001063
employment_population_ratio                      0.001063
nonfarm_payroll_employment                       0.000000
industrial_production_index                      0.000000
dtype: float64

Looks like we have a ton of missing values. Let's investigate the causes of this 'missingness' - we could be dealing with data reported at a different frequency (e.g. quarterly instead of annually) or perhaps certain metrics only started being recorded later on (e.g. 1960's instead of in 1948).

In [ ]:
raw_df.apply(lambda col: col.first_valid_index()).sort_values()

unemployment_rate                               1948-01-01
labor_force_participation_rate                  1948-01-01
nonfarm_payroll_employment                      1948-01-01
employment_population_ratio                     1948-01-01
consumer_price_index                            1948-01-01
industrial_production_index                     1948-01-01
gdp                                             1948-01-01
federal_funds_rate                              1954-07-01
core_consumer_price_index                       1957-01-01
personal_consumption_expenditures_price_index   1959-01-01
housing_starts                                  1959-01-01
ten_year_treasury_yield                         1962-02-01
initial_jobless_claims                          1967-04-01
two_year_treasury_yield                         1976-06-01
retail_sales                                    1992-01-01
sp500_index                                     2016-07-01
dtype: datetime64[us]

Looking at where each field has it's first valid entry:
1. sp500_index, retail_sales don't have valid values until 1992/2016 respectively, we will remove these variables
2. initial_jobless_claims is missing in about 89% of rows which makes it functionally unusable - we will also remove this variable

Removing those three sparse, unusable variables leaves us with three variables with many missing values (>40%)
1. gdp
2. ten_year_treasury_yield
3. two_year_treasury_yield

In [14]:
def print_index_gaps(df, columns):
    """
    Prints column names and the distribution of index time gaps between observations.
    """
    for col in columns:
        print(col)

        s = df[col].dropna()
        diffs = s.index.to_series().diff().value_counts()

        print(diffs.head())
        print()

print_index_gaps(raw_df, ['gdp', 'ten_year_treasury_yield', 'two_year_treasury_yield'])

gdp
92 days    156
91 days     98
90 days     58
Name: count, dtype: int64

ten_year_treasury_yield
61 days    134
31 days    126
30 days    101
62 days     54
28 days     35
Name: count, dtype: int64

two_year_treasury_yield
61 days    103
31 days     97
30 days     80
62 days     43
28 days     27
Name: count, dtype: int64



We suspect gdp nulls may be attributed to quarterly reporting (instead of monthly). We also want to investigate if there are patterns to missing values for the treasury yield variables. Lets use 'print_index_gaps' for this analysis.

1. gdp observations are spaced out between 90-92 days, quarterly reporting. We can forward fill these values - as GDP is only known when released each quarter - between releases the latest values is the current best estimate.
2. ten_year_treasury_yield/two_year_treasury_yield look to be reported monthly with some missing months. We can use time interpolation here by making a market assumption of smooth curves between observations. We could also foward fill these as well.

In [18]:
df = raw_df.copy()

# Ensure datetime index
df.index = pd.to_datetime(df.index)
df = df.sort_index()

# Keep only 1959-01-01 onward
df = df.loc[df.index >= "1959-01-01"]

# Drop unwanted columns (ignore errors in case any are missing)
df = df.drop(
    columns=[
        "sp500_index",
        "retail_sales",
        "initial_jobless_claims"
    ],
    errors="ignore"
)

# GDP: forward fill
df['gdp'] = df['gdp'].ffill()

# Treasury yields: time interpolation
df['ten_year_treasury_yield'] = df['ten_year_treasury_yield'].interpolate(method='time')
df['two_year_treasury_yield'] = df['two_year_treasury_yield'].interpolate(method='time')

In [23]:
print(df.apply(lambda col: col.first_valid_index()).sort_values())
print(df.isna().mean().sort_values(ascending=False))

unemployment_rate                               1959-01-01
labor_force_participation_rate                  1959-01-01
nonfarm_payroll_employment                      1959-01-01
employment_population_ratio                     1959-01-01
consumer_price_index                            1959-01-01
core_consumer_price_index                       1959-01-01
personal_consumption_expenditures_price_index   1959-01-01
federal_funds_rate                              1959-01-01
industrial_production_index                     1959-01-01
housing_starts                                  1959-01-01
gdp                                             1959-01-01
ten_year_treasury_yield                         1962-02-01
two_year_treasury_yield                         1976-06-01
dtype: datetime64[us]
two_year_treasury_yield                          0.258344
ten_year_treasury_yield                          0.045735
employment_population_ratio                      0.001236
labor_force_participation_rate       

We will use these three modeling approaches to attempt to predict unemployment rate:
1. Linear Regression (baseline)
2. Random Forest
3. XGBoost